# GTEx CLAMP models with all pathways prior (all samples)

**Environment:** `clamp-analyses`

Runs CLAMPfull with all pathways prior (Hallmark, Reactome, GO CC, C8) using all GTEx samples, reusing the FBM, SVD, and CLAMPbase results already generated in `nbs/01_model_building/02_gtex/01_CLAMP.ipynb`.

Steps:
1. Load existing FBM, SVD, CLAMPbase, and CLAMP_K from `config$GTEx$OUTPUT_DIR`
2. Run CLAMPfull with the combined all-pathways prior
3. Save results to `config$GTEx$OUTPUT_DIR/CLAMPfull_hall`

## Load libraries

In [ ]:
library(bigstatsr)
library(data.table)
library(dplyr)
library(Matrix)
library(here)
library(CLAMP)

source(here("config.R"))

## Configuration

In [ ]:
output_data_dir <- config$GTEx$OUTPUT_DIR
pathways_path   <- here::here('data/pathways')

MULTIPLIER <- 100
MAX_ITER   <- 5000

message("GTEx output dir: ", output_data_dir)

## Load pre-built GTEx inputs

In [ ]:
gtex_genes  <- readRDS(file.path(output_data_dir, "gtex_genes.rds"))
samples     <- readRDS(file.path(output_data_dir, "gtex_samples.rds"))
gtex_fbm_filt <- readRDS(file.path(output_data_dir, "gtex_fbm_filt.rds"))
gtex_svdRes   <- readRDS(file.path(output_data_dir, "gtex_svdRes.rds"))
gtex_baseRes  <- readRDS(file.path(output_data_dir, "CLAMPbase.rds"))
CLAMP_K_gtex  <- readRDS(file.path(output_data_dir, "CLAMP_K_gtex.rds"))

message("Genes: ", length(gtex_genes))
message("Samples: ", length(samples))
message("CLAMP K: ", CLAMP_K_gtex)

## Load and match all pathways prior

In [ ]:
hall_gmt     <- CLAMP:::read_gmt(file.path(pathways_path, "h.all.v2026.1.Hs.symbols.gmt"))
reactome_gmt <- CLAMP:::read_gmt(file.path(pathways_path, "c2.cp.reactome.v2026.1.Hs.symbols.gmt"))
gocc_gmt     <- CLAMP:::read_gmt(file.path(pathways_path, "c5.go.cc.v2026.1.Hs.symbols.gmt"))
c8_gmt       <- CLAMP:::read_gmt(file.path(pathways_path, "c8.all.v2026.1.Hs.symbols.gmt"))

names(hall_gmt)     <- paste0("HALL_",     names(hall_gmt))
names(reactome_gmt) <- paste0("REACTOME_", names(reactome_gmt))
names(gocc_gmt)     <- paste0("GOCC_",     names(gocc_gmt))
names(c8_gmt)       <- paste0("C8_",       names(c8_gmt))

all_pathways_list <- list(
  HALL     = hall_gmt,
  REACTOME = reactome_gmt,
  GOCC     = gocc_gmt,
  C8       = c8_gmt
)

all_pathways_pathMat <- gmtListToSparseMat(all_pathways_list)
all_pathways_matched <- getMatchedPathwayMat(all_pathways_pathMat, gtex_genes)
message("Loaded and matched all pathways matrix against GTEx genes")

## Run CLAMPfull with all pathways prior

In [ ]:
message("Running CLAMPfull with all pathways prior on GTEx (all samples)...")

gtex_fullRes_hall <- CLAMPfull(
  Y                 = gtex_fbm_filt,
  svdres            = gtex_svdRes,
  priorMat          = all_pathways_matched,
  clamp.base.result = gtex_baseRes,
  use_cpp           = TRUE,
  trace             = TRUE,
  multiplier        = MULTIPLIER,
  max.iter          = MAX_ITER,
  clamp_k           = CLAMP_K_gtex
)

gtex_fullRes_hall$Z <- data.frame(gtex_fullRes_hall$Z)
rownames(gtex_fullRes_hall$Z) <- gtex_genes

gtex_fullRes_hall$B <- data.frame(gtex_fullRes_hall$B)
colnames(gtex_fullRes_hall$B) <- samples

gtex_fullRes_hall$summary <- gtex_fullRes_hall$summary %>%
  dplyr::rename(LV = LV_index) %>%
  dplyr::mutate(LV = paste0('LV', LV))

message("CLAMPfull completed")

## Save results

In [ ]:
dst_dir <- file.path(output_data_dir, "CLAMPfull_hall")
dir.create(dst_dir, showWarnings = FALSE, recursive = TRUE)

saveRDS(gtex_fullRes_hall, file = file.path(output_data_dir, "CLAMPfull_hall.rds"))

write.csv(gtex_fullRes_hall$B,       file.path(dst_dir, "B.csv"))
write.csv(gtex_fullRes_hall$Z,       file.path(dst_dir, "Z.csv"))
write.csv(gtex_fullRes_hall$summary, file.path(dst_dir, "summary.csv"))

message("Results saved to: ", dst_dir)